<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/dnn-course-2026-1/blob/main/C1M1_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Atividade Avaliativa - Predição Inteligente de Logística Urbana

Nesse módulo você passou de modelos lineares simples para redes que conseguem capturar padrões complexos e não lineares. Agora, é hora de aplicar essas habilidades em um desafio que reflete como os projetos funcionam em um cenário do mundo real.

Até agora, você trabalhou com tensores pequenos, criados manualmente. Desta vez, você subirá de nível ao carregar um conjunto de dados maior a partir de um arquivo `.csv`, um primeiro passo comum em qualquer tarefa de aprendizado de máquina. Este problema também é mais complexo: em vez de uma única entrada prevendo um resultado, você terá **múltiplos atributos** (features) que trabalham juntos para influenciar o tempo de entrega final.

Esta tarefa também apresenta uma das partes mais criativas e impactantes do aprendizado de máquina: a **engenharia de atributos** (feature engineering). Você escreverá uma função que cria um atributo completamente novo a partir dos dados existentes. Projetar atributos como esse é uma habilidade importante que permite construir modelos mais poderosos e perspicazes.

**O Que Você Fará Nesta Tarefa**

* Preparar o conjunto de dados de múltiplos atributos usando normalização e manipulações avançadas de tensores.
* Criar um novo atributo para capturar padrões mais complexos.
* Construir uma rede neural mais sofisticada com múltiplas camadas ocultas.
* Treinar seu modelo nos dados preparados.
* Prever o tempo de entrega para um pedido novo e nunca antes visto.

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">DICAS PARA O SUCESSO NA AVALIAÇÃO DA SUA TAREFA:</h4>

* Todas as células estão bloqueadas, exceto aquelas onde você deve inserir suas soluções ou quando for explicitamente mencionado que você pode interagir com elas.

* Em cada célula de exercício, procure pelos comentários `### COMECE SEU CÓDIGO ###` e `### TERMINE SEU CÓDIGO AQUI ###`. Eles indicam onde você deve escrever o código da solução. **Não adicione ou altere nenhum código que esteja fora desses comentários**.

* Você pode adicionar novas células para experimentar, mas elas serão ignoradas durante a avaliação; portanto, não dependa de células recém-criadas para hospedar seu código de solução, use os locais fornecidos para isso.

* Evite o uso de variáveis globais, a menos que seja absolutamente necessário. O avaliador testa seu código em um ambiente isolado sem executar todas as células desde o topo. Como resultado, variáveis globais podem estar indisponíveis no momento da correção. Variáveis globais que devem ser utilizadas estarão definidas em MAIÚSCULAS.
---

## Sumário
- Importar bibliotecas necessárias
- 1 - Dados de Múltiplos Atributos
    - 1.1 - Carregando e Explorando os Dados Brutos
    - 1.2 - Engenharia de Atributos: Adicionando Horário de Pico
        - **Exercício 1 - rush_hour_feature**
    - 1.3 - Construindo o Pipeline de Preparação de Dados
        - **Exercício 2 - prepare_data**
    - 1.4 - Visualizando os Dados Preparados
- 2 - Construindo a Rede Neural
    - **Exercício 3 - init_model**
- 3 - Treinando o Modelo
    - **Exercício 4 - train_model**
- 4 - Avaliando o Desempenho do Modelo
- 5 - Realizando uma Nova Predição

<a name='0'></a>
## Importar bibliotecas necessárias

In [ ]:
import os

def download_files(filenames):
    """
    Baixa uma lista de arquivos do repositório do curso se eles não existirem localmente.
    """
    base_url = "https://raw.githubusercontent.com/fabiobento/dnn-course-2026-1/main/"
    
    for filename in filenames:
        if not os.path.exists(filename):
            url = f"{base_url}{filename}"
            print(f"Baixando {filename}...")
            # Usando o comando do sistema para baixar
            os.system(f"wget -q {url} -O {filename}")
        else:
            print(f"O arquivo '{filename}' já existe localmente. Pulando...")

# Arquivos auxiliares necessários para esse notebook:
arquivos_necessários = [
    'helper_utils_2.py',
    'data_with_features.csv',
    'unittests_utils.py',
    'unittests.py'
]

download_files(arquivos_necessários)

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
import matplotlib.pyplot as plt

import helper_utils_2
import unittests

<a name='1'></a>
## 1 - Dados de Múltiplos Atributos

Desta vez, você trabalhará com um conjunto de dados muito mais rico vindo de um arquivo `.csv`, contendo registros de **100 entregas passadas**. Diferente dos laboratórios anteriores, onde o tempo dependia apenas da distância, este novo problema é mais complexo. O tempo final de entrega agora é influenciado por múltiplos atributos de entrada.

Aqui está o detalhamento dos dados com os quais você trabalhará:

* **distance_miles**: A distância total da rota de entrega em **milhas**, representada como um número de ponto flutuante.

* **time_of_day_hours**: O horário em que o pedido foi **despachado para entrega** em **horas**, em um relógio de 24 horas, representado como um número de ponto flutuante (ex: `16.07` representa um horário de despacho logo após as 16:00).

* **is_weekend**: Um atributo binário que representa o dia da semana, onde `1` indica final de semana e `0` indica dia útil.

* **delivery_time_minutes**: Esta é a sua **variável alvo**. É o tempo total que a entrega levou em minutos, representado como um número de ponto flutuante.

Para tornar o cenário mais realista, estes dados operam sob algumas regras de negócio: **as entregas ocorrem apenas entre 8:00 (8.0) e 20:00 (20.0), e a empresa não realiza entregas a mais de 20 milhas de distância**.

<a name='1-1'></a>
### 1.1 - Carregando e Explorando os Dados Brutos

Carregue e entenda seus dados.

* Defina o caminho do arquivo para o seu conjunto de dados, `./data_with_features.csv`.
* Use a biblioteca Pandas para carregar o conjunto de dados do caminho de arquivo fornecido como um DataFrame, `data_df`, uma estrutura poderosa para manipular e analisar dados.
* Inspecione o formato (*shape*) dos seus dados, que aparecerá como **100 linhas** (representando 100 entregas) e **4 colunas**.

In [ ]:
# Carregar o conjunto de dados a partir do arquivo CSV
file_path = './data_with_features.csv'
data_df = pd.read_csv(file_path)

# Imprimir o formato (shape) do DataFrame
print(f"Formato do Conjunto de Dados: {data_df.shape}\n")

* Inspecione as linhas do conjunto de dados carregado.
    * Por padrão, `rows_to_display` está definido como `10`, mas sinta-se à vontade para alterar esse valor para explorar melhor os dados.

In [ ]:
# CÉLULA EDITÁVEL:

# Defina o número de linhas que você deseja exibir.
rows_to_display = 10

# Exibir as linhas
print(data_df.head(rows_to_display))

<br>

Agora que os dados foram carregados, é hora de visualizá-los para entender as relações entre seus atributos e o que você está tentando prever.

A função auxiliar `plot_delivery_data` abaixo criará um gráfico de dispersão detalhado que visualiza todos os quatro atributos de uma vez:

* O **eixo x** representará a **distância** da entrega.
* O **eixo y** representará o **tempo de entrega**.
* A **cor** de cada ponto mostrará o **horário do dia**, com cores mais claras para despachos mais cedo e tons de vermelho mais escuros para os mais tardios.
* O **estilo** de cada ponto indicará o tipo de dia, com círculos preenchidos para dias úteis e círculos vazios para fins de semana.

Procure por padrões no gráfico. Você consegue perceber como diferentes atributos podem estar influenciando o tempo de entrega?

In [ ]:
helper_utils_2.plot_delivery_data(data_df)

<a name='1-2'></a>
### 1.2 - Engenharia de Atributos: Adicionando Horário de Pico

A visualização acima revela um padrão interessante: algumas entregas levam mais tempo mesmo para a mesma distância, provavelmente devido ao tráfego intenso durante os **horários de pico**.

Em vez de esperar que o modelo aprenda esse padrão complexo por conta própria, você pode usar a **engenharia de atributos** (feature engineering). Esta é uma etapa criativa onde você aplica conhecimento de domínio para tornar esses padrões explícitos. Você criará um novo atributo que informa diretamente ao modelo quando uma entrega ocorre dentro de uma janela de horário de pico.

> Este novo atributo será `1` se uma entrega foi despachada durante o pico da manhã `(08:00 - 10:00)` ou o pico da tarde `(16:00 - 19:00)` em um **dia útil**, e `0` caso contrário.

Agora, você deve estar se perguntando: por que o horário de pico só é considerado em dias úteis? Isso reflete um padrão comum do mundo real. O conceito de "hora do rush" está tradicionalmente ligado ao tráfego de pessoas indo e voltando do trabalho nos dias de semana, que é o padrão que mais impacta os tempos de entrega de forma previsível em escala urbana. Esse padrão específico desaparece nos fins de semana. Portanto, é uma premissa realista de que o principal causador de atrasos no horário de pico seja o deslocamento em dias úteis.

Antes de aplicar a lógica a todo o conjunto de dados, é uma boa prática trabalhar com uma pequena amostra. Isso permite que você construa e teste sua função rapidamente.

* Defina as primeiras 5 linhas do seu `data_df` como um **tensor do PyTorch**.
* Você utiliza um tensor para esta amostra porque seu conjunto de dados completo também será carregado como um tensor. Isso garante que a função que você construir agora funcionará no conjunto de dados completo mais tarde, sem qualquer alteração.
* Este tensor inicial contém todos os dados para cada entrega da amostra.

In [ ]:
# Definir as 5 linhas de dados como um único tensor 2D
sample_tensor = torch.tensor([
    # distância, hora_do_dia, é_fim_de_semana, tempo_de_entrega
    [1.60,      8.20,        0,          7.22],   # linha 1
    [13.09,     16.80,       1,          32.41],  # linha 2       
    [6.97,      8.02,        1,          17.47],  # linha 3
    [10.66,     16.07,       0,          37.17],  # linha 4
    [18.24,     13.47,       0,          38.36]   # linha 5
], dtype=torch.float32)

* Para criar o atributo de pico (rush feature), seu cálculo depende apenas do **horário do dia** e se é um **dia útil**.
* Use a operação de fatiamento (slicing) de tensores para selecionar apenas essas duas colunas, ignorando os dados desnecessários de distância e tempo de entrega para esta etapa.
    * O time_of_day_hours está no índice de coluna 1, e o is_weekend está no índice de coluna 2.

In [ ]:
# Usar fatiamento de tensor para separar cada coluna
# A sintaxe de fatiamento é [:, indice_da_coluna]
sample_hours = sample_tensor[:, 1]
sample_weekends = sample_tensor[:, 2]

print("--- Tensores Fatiados ---")
print(f"Horas da Amostra:       {sample_hours}")
print(f"Finais de Semana (Amostra): {sample_weekends}\n")

<br>

Agora que você tem os tensores `sample_hours` e `sample_weekends` preparados, você os usará para construir a função `rush_hour_feature`.

<a name='ex-1'></a>
### Exercício 1 - rush_hour_feature

Implemente a função `rush_hour_feature`.

**Sua Tarefa:**

* **Defina as condições individuais:**
    * Defina `is_morning_rush` como `True` onde o `hours_tensor` for maior ou igual a `8.0` **E** menor que `10.0`.
    * Defina `is_evening_rush` como `True` onde o `hours_tensor` for maior ou igual a `16.0` **E** menor que `19.0`.
    * Defina `is_weekday` como `True` onde o `weekends_tensor` for igual a `0`.
>
* **Combine as condições:**
    * Defina `is_rush_hour_mask` combinando os três tensores booleanos. A lógica deve ser `True` apenas se for um dia útil **E** (for pico da manhã **OU** pico da tarde).

**Dica**: Você pode usar operadores de comparação padrão (`>=`, `<`, `==`) e operadores lógicos como `&` (E) e `|` (OU) diretamente em tensores do PyTorch.

<details>
  <summary><b><font color="green">Dicas Adicionais de Código (Clique para expandir se estiver travado)</font></b></summary>
  
Se você estiver com dificuldades, pense em como construir cada máscara booleana passo a passo.

**Para `is_morning_rush`:**
* Isso requer duas comparações no `hours_tensor` unidas por um **E** lógico (`&`).
* Por exemplo, a primeira parte da condição para o início da janela é `(hours_tensor >= 8.0)`. Você precisará criar a segunda condição (`< 10.0`) e combiná-las.

**Para `is_evening_rush`:**
* Aplique a mesma lógica do pico da manhã, mas para a janela de tempo da tarde.
* Por exemplo, a segunda parte da condição para o fim da janela é `(hours_tensor < 19.0)`. Você precisará criar a primeira condição (`>= 16.0`) e combiná-las.

**Para `is_weekday`:**
* Esta é uma comparação única. Você precisa verificar quais elementos no `weekends_tensor` são iguais (`==`) a `0`. A lógica é: `(weekends equals 0)`.

**Para `is_rush_hour_mask`:**
* Esta etapa combina as três variáveis que você acabou de criar.
* A lógica é: `dia útil E (pico da manhã OU pico da tarde)`.
* Lembre-se de usar parênteses `()` para agrupar as condições `is_morning_rush` e `is_evening_rush` com o operador **OU** lógico (`|`).

</details>

In [ ]:
# FUNÇÃO AVALIADA (GRADED FUNCTION): rush_hour_feature

def rush_hour_feature(hours_tensor, weekends_tensor):
    """
    Cria uma nova característica binária indicando se uma entrega ocorre no horário de pico em dia útil.

    Argumentos:
        hours_tensor (torch.Tensor): Um tensor com os horários das entregas.
        weekends_tensor (torch.Tensor): Um tensor indicando se a entrega é no fim de semana.

    Retorna:
        torch.Tensor: Um tensor de 0s e 1s indicando horário de pico em dia útil.
    """

    ### COMECE SEU CÓDIGO AQUI ###
    
    # Definir as condições de horário de pico e dia útil
    is_morning_rush = None
    is_evening_rush = None
    is_weekday = None

    # Combinar as condições para criar a máscara final de horário de pico
    is_rush_hour_mask = None

    ### FIM DO SEU CÓDIGO AQUI ###

    # Converter a máscara booleana em um tensor float para usar como característica numérica
    return is_rush_hour_mask.float()

In [ ]:
rush_hour_for_sample = rush_hour_feature(sample_hours, sample_weekends)

print(f"Horas da Amostra:       {sample_hours.numpy()}")
print(f"Amostra de Fins de Semana Amostra: {sample_weekends.numpy()}")
print(f"É Hora de Pico?:        {rush_hour_for_sample.numpy()}")

#### Saída Esperada

```
Horas da Amostra:     [ 8.2  16.8   8.02 16.07 13.47]
Amostra de Fins de Semana Amostra:  [0. 1. 1. 0. 0.]
É Hora de Pico?:    [1. 0. 0. 1. 0.]
```

In [ ]:
# Teste seu código!
unittests.exercise_1(rush_hour_feature)

<a name='1-3'></a>
### 1.3 - Construindo o Pipeline de Preparação de Dados

Agora que você tem sua função de engenharia de atributos, você a aplicará ao pipeline de preparação de dados. O objetivo é criar uma única função que receba o DataFrame bruto do Pandas como entrada e gere os tensores finais de `features` (atributos) e `targets` (alvos) que seu modelo usará para o treinamento.

Esta função realizará várias transformações importantes: ela chamará sua função `rush_hour_feature()` para adicionar o novo atributo criado, normalizará as colunas `distance_miles` e `time_of_day_hours` para que estejam em uma escala comparável e lidará com todas as operações de tensores necessárias para estruturar os dados corretamente.

Este processo resultará em um único tensor de `features` e um único tensor de `targets`, perfeitamente formatados para sua rede neural.

<a name='ex-2'></a>
### Exercício 2 - prepare_data

**Sua Tarefa**:

Sua tarefa é implementar as etapas principais de manipulação de tensores dentro da função `prepare_data`. O código para normalização e combinação dos atributos finais já foi fornecido.

* **Converter DataFrame em Tensor**:
    * Converta os `all_values` (que são extraídos do DataFrame do Pandas) em um único tensor do PyTorch, `full_tensor`.
    * Lembre-se de definir o `dtype` como `torch.float32`.
>    
* **Fatiar em Tensores Brutos**:
    * Use o fatiamento (slicing) de tensores para separar o `full_tensor` em tensores 1D individuais para cada coluna:
        * `raw_distances` (do índice de coluna 0)
        * `raw_hours` (do índice de coluna 1)
        * `raw_weekends` (do índice de coluna 2)
        * `raw_targets` (do índice de coluna 3)
>        
* **Criar o Atributo de Engenharia**:
    * Chame a função `rush_hour_feature()` que você acabou de construir.
    * Passe seus tensores recém-fatiados `raw_hours` e `raw_weekends` para ela.
>    
* **Remodelar Tensores de Atributos**:
    * Use o método `.unsqueeze(1)` em cada um dos seus quatro tensores de atributos (`raw_distances`, `raw_hours`, `raw_weekends` e `is_rush_hour_feature`) para adicionar uma nova dimensão.

<details>
  <summary><b><font color="green">Dicas Adicionais de Código (Clique para expandir se estiver travado)</font></b></summary>
  
Se precisar de uma ajudinha, aqui está um guia mais detalhado para cada etapa dentro da função.

**Para `full_tensor`:**
* Você usará a função `torch.tensor()` aqui.
* O primeiro argumento deve ser `all_values`, e você também deve definir o `dtype` como `torch.float32`.

**Para fatiar nos tensores `raw_`:**
* Você usará a sintaxe de fatiamento `full_tensor[:, indice]` para cada variável.
* Por exemplo, para obter `raw_distances`, o código seria `raw_distances = full_tensor[:, 0]`. Siga este padrão para as outras três variáveis usando seus respectivos índices de coluna.

**Para `is_rush_hour_feature`:**
* Esta etapa é apenas uma chamada de função.
* Você precisa chamar `rush_hour_feature()` e passar os dois tensores que ela necessita: `raw_hours` e `raw_weekends`.

**Para remodelar tensores de atributos (ex: `distances_col`):**
* Este é um passo crucial para deixar seus dados prontos para o modelo.
* Para cada um dos quatro tensores de atributos que você possui (`raw_distances`, `raw_hours`, etc.), você precisa chamar o método `.unsqueeze(1)` nele.
* Por exemplo, o primeiro seria `distances_col = raw_distances.unsqueeze(1)`.

</details>

In [3]:
# FUNÇÃO AVALIADA: prepare_data

def prepare_data(df):
    """
    Converte um DataFrame do pandas em tensores PyTorch preparados para modelagem.

    Argumentos:
        df (pd.DataFrame): Um DataFrame pandas contendo os dados brutos de entrega.

    Retorna:
        prepared_features (torch.Tensor): O tensor de atributos (features) 2D final para o modelo.
        prepared_targets (torch.Tensor): O tensor de alvos (targets) 2D final.
        results_dict (dict): Um dicionário de tensores intermediários para fins de teste.
    """

    # Extrai os dados do DataFrame como uma matriz NumPy
    # (Não existe um torch.from_dataframe() direto, então usamos .values para obter uma matriz NumPy primeiro)
    all_values = df.values

    ### INICIE O CÓDIGO AQUI ###

    # Converta todos os valores do DataFrame em um único tensor PyTorch
    full_tensor = None

    # Use o fatiamento de tensores para separar cada coluna bruta
    raw_distances = None
    raw_hours = None
    raw_weekends = None
    raw_targets = None

    # Chame sua função rush_hour_feature() para gerar o novo atributo
    is_rush_hour_feature = None

    # Use o método .unsqueeze(1) para remodelar os quatro tensores de atributos 1D em vetores de coluna 2D
    distances_col = None
    hours_col = None
    weekends_col = None
    rush_hour_col = None

    ### FIM DO CÓDIGO AQUI ###

    # Normaliza as colunas de atributos contínuos (distância e tempo)
    dist_mean, dist_std = distances_col.mean(), distances_col.std()
    hours_mean, hours_std = hours_col.mean(), hours_col.std()
 
    distances_norm = (distances_col - dist_mean) / dist_std
    hours_norm = (hours_col - hours_mean) / hours_std

    # Combina todos os atributos 2D preparados em um único tensor
    prepared_features = torch.cat([
        distances_norm,
        hours_norm,
        weekends_col,
        rush_hour_col
    ], dim=1) # dim=1 os concatena por coluna, empilhando os atributos lado a lado

    # Prepara os alvos garantindo que eles tenham o formato (shape) correto
    prepared_targets = raw_targets.unsqueeze(1)
    
    # Dicionário para Fins de Teste
    results_dict = {
        'full_tensor': full_tensor,
        'raw_distances': raw_distances,
        'raw_hours': raw_hours,
        'raw_weekends': raw_weekends,
        'raw_targets': raw_targets,
        'distances_col': distances_col,
        'hours_col': hours_col,
        'weekends_col': weekends_col,
        'rush_hour_col': rush_hour_col
    }
    

    return prepared_features, prepared_targets, results_dict

In [ ]:
# Cria um pequeno DataFrame de teste com as primeiras 5 entradas
test_df = data_df.head(5).copy()

# Exibe o estado "Antes" como um tensor bruto
raw_test_tensor = torch.tensor(test_df.values, dtype=torch.float32)
print("--- Tensor Bruto (Antes da Preparação) ---\n")
print(f"Formato (Shape): {raw_test_tensor.shape}")
print("Valores:\n", raw_test_tensor)
print("\n" + "="*50 + "\n")

# Executa a função para obter os tensores "depois" de preparados
test_features, test_targets, _ = prepare_data(test_df)

# Exibe o estado "Depois"
print("--- Tensores Preparados (Após a Preparação) ---")
print("\n--- Atributos (Features) Preparados ---\n")
print(f"Formato (Shape): {test_features.shape}")
print("Valores:\n", test_features)

print("\n--- Alvos (Targets) Preparados ---")
print(f"Formato (Shape): {test_targets.shape}")
print("Valores:\n", test_targets)

#### Saída Esperada

```
--- Tensores Preparados (Após a Preparação) ---

--- Recursos (Features) Preparados ---

Shape: torch.Size([5, 4])
Values:
 tensor([[-1.3562, -1.0254,  0.0000,  1.0000],
        [ 0.4745,  1.0197,  1.0000,  0.0000],
        [-0.5006, -1.0682,  1.0000,  0.0000],
        [ 0.0873,  0.8461,  0.0000,  1.0000],
        [ 1.2951,  0.2278,  0.0000,  0.0000]])

--- Alvos (Targets) Preparados ---
Shape: torch.Size([5, 1])
Values:
 tensor([[ 7.2200],
        [32.4100],
        [17.4700],
        [37.1700],
        [38.3600]])
```

In [ ]:
# Teste seu código!
unittests.exercise_2(prepare_data)

<br>

Excelente! Como você pode ver pelos resultados da amostra acima, sua função `prepare_data` transformou com sucesso os dados brutos nos dois tensores distintos que seu modelo precisa para o treinamento.

Você começou com um arquivo `.csv` contendo todos os dados de cada entrega. Sua função processou isso e produziu:

* **Um tensor de "features" (atributos)**: Contém as quatro colunas de dados de entrada (distância, hora do dia, indicador de fim de semana, indicador de horário de pico) com as quais o modelo aprenderá. Observe como as duas primeiras colunas foram normalizadas e a quarta coluna é o seu novo atributo criado, `is_rush_hour`.

* **Um tensor de "targets" (alvos)**: Contém apenas o `delivery_time_minutes` final, separado dos atributos de entrada. Este é o valor que seu modelo aprenderá a prever.

Agora que você verificou que seu pipeline de preparação de dados funciona corretamente em uma pequena amostra, é hora de executá-lo em todo o conjunto de dados para preparar todos os 100 registros de entrega para o treinamento.

In [ ]:
# Processa o DataFrame inteiro para obter os tensores finais de atributos (features) e alvos (targets).
features, targets, _ = prepare_data(data_df)

<a name='1-4'></a>
### 1.4 - Visualizando os Dados Preparados

Agora que seu pipeline de preparação de dados está completo, você pode visualizar os resultados para confirmar se sua engenharia de atributos funcionou como esperado.

#### Gráfico de Entregas em Horário de Pico

* Execute a célula abaixo para exibir um gráfico de dispersão, mostrando a relação entre o `Tempo de Entrega` e a `Distância`.
* Os pontos serão coloridos com base no seu novo atributo, facilitando a distinção entre entregas em "Horário de Pico" e "Fora do Horário de Pico".

In [ ]:
helper_utils_2.plot_rush_hour(data_df, features)

#### Gráfico Final dos Dados Preparados

* Execute a célula abaixo para exibir o gráfico de dispersão que visualiza os dados finais que você usará para treinar seu modelo. Ele mostrará o `Tempo de Entrega` vs. `Distância Normalizada`.
* Os pontos são estilizados em quatro categorias, combinando o tipo de dia e seu novo atributo de horário de pico.
* Observe que "Final de Semana (Horário de Pico)" não aparece, pois seu atributo se aplica corretamente apenas aos dias úteis (conforme explicado anteriormente).

In [ ]:
helper_utils_2.plot_final_data(features, targets)

<a name='2'></a>
## 2 - Construindo a Rede Neural

Com seu pipeline de dados concluído, você está pronto para a próxima grande etapa: **construir o modelo.**

Como seu problema agora envolve múltiplos atributos, você precisará de uma arquitetura mais sofisticada do que as que viu anteriormente. Você construirá uma rede neural com duas camadas ocultas para capturar as relações complexas entre todos os seus atributos de entrada.

<a name='ex-3'></a>
### Exercício 3 - init_model

Implemente a função `init_model` para definir a arquitetura do modelo, o otimizador e a função de perda (*loss function*).

**Sua Tarefa:**

* **Definir a Arquitetura do Modelo**:
    * Defina um `model` usando `nn.Sequential`.
    * O modelo **deve ter apenas** três camadas `nn.Linear`, cada uma seguida por uma função de ativação `nn.ReLU()`, exceto a última.
        * **Camada de Entrada**: Uma camada `nn.Linear` que aceita **4 atributos de entrada** e gera **64 atributos**.
        * **Camada Oculta**: Uma camada `nn.Linear` que recebe os **64 atributos** da camada anterior e gera **32 atributos**.
        * **Camada de Saída**: Uma camada `nn.Linear` final que recebe os **32 atributos** da camada oculta e produz um **único valor** de saída.
>        
* **Definir o Otimizador**:
    * Defina o `optimizer` como **Gradiente Descendente Estocástico (SGD)**. Você precisa passar os parâmetros do modelo (`model.parameters()`) e definir a taxa de aprendizado (`lr`) como `0.01`.
>
* **Definir a Função de Perda**:
    * Defina a `loss_function` como o **Erro Quadrático Médio (MSE)**.

<details>
  <summary><b><font color="green">Dicas Adicionais de Código (Clique para expandir se estiver travado)</font></b></summary>
  
**Para o Modelo:**
* Lembre-se de listar suas camadas dentro do construtor `nn.Sequential()`, separadas por vírgulas.
* A camada `nn.Linear()` recebe dois argumentos principais: `in_features` e `out_features`. Certifique-se de que o `in_features` de uma camada corresponda ao `out_features` da camada anterior.
* A ordem correta das camadas é: **Camada de Entrada -> ReLU -> Camada Oculta -> ReLU -> Camada de Saída.**

**Para o Otimizador:**
* Você usará `optim.SGD`. Seu primeiro argumento são os parâmetros do modelo, que você obtém com `model.parameters()`.
* O segundo argumento que você precisa fornecer é a taxa de aprendizado, `lr=0.01`.

**Para a Função de Perda:**
* Você usará `nn.MSELoss`. Como é uma classe, você precisa criar uma instância dela chamando-a com parênteses: `nn.MSELoss()`.

</details>

In [ ]:
# FUNÇÃO AVALIADA: init_model

def init_model():
    """
    Inicializa o modelo de rede neural, o otimizador e a função de perda.

    Retorna:
        model (nn.Sequential): O modelo sequencial PyTorch inicializado.
        optimizer (torch.optim.Optimizer): O otimizador inicializado para o treinamento.
        loss_function: A função de perda inicializada.
    """

    # Define a semente aleatória para reprodutibilidade dos resultados (NÃO A MANIPULE)
    torch.manual_seed(41)

    ### INICIE O CÓDIGO AQUI ###

    # Define a arquitetura do modelo usando nn.Sequential
    model = None(
        # Camada de entrada (Linear): 4 atributos de entrada, 64 atributos de saída
        None,
        # Primeira função de ativação ReLU
        None,
        # Camada oculta (Linear): 64 entradas, 32 saídas
        None,
        # Segunda função de ativação ReLU
        None,
        # Camada de saída (Linear): 32 entradas, 1 saída (a previsão)
        None,
    ) 
    
    # Define o otimizador (Gradiente Descendente Estocástico - SGD)
    optimizer = None(None, None)

    # Define a função de perda (Erro Quadrático Médio - MSE para regressão)
    loss_function = None

    ### FIM DO CÓDIGO AQUI ###

    return model, optimizer, loss_function

In [ ]:
model, optimizer, loss_function = init_model()

print(f"{'='*30}\nArquitetura do Modelo Inicializada\n{'='*30}\n{model}")
print(f"\n{'='*30}\nOtimizador\n{'='*30}\n{optimizer}")
print(f"\n{'='*30}\nFunção de Perda (Loss)\n{'='*30}\n{loss_function}")

#### Saída Esperada:

```
==============================
Arquitetura do Modelo Inicializado
==============================
Sequential(
  (0): Linear(in_features=4, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)

==============================
Otimizador
==============================
SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

==============================
Função de Perda
==============================
MSELoss()
```

In [ ]:
# Teste seu código!
unittests.exercise_3(init_model)

<a name='3'></a>
## 3 - Treinando o Modelo

Com seus dados preparados e a arquitetura do modelo definida, é hora da etapa mais importante: o **treinamento**. 

<a name='ex-4'></a>
### Exercício 4 - train_model

Implemente o loop de treinamento completo dentro da função `train_model`.

**Sua Tarefa:**

* **Inicialize seu modelo e ferramentas:**
    * Comece chamando a função `init_model()` que você construiu anteriormente para obter seu `model`, `optimizer` e `loss_function`.
>    
* **Itere pelas épocas:**
    * Crie um loop `for` que itere de 0 até o número de épocas (`epochs`) fornecido.
>    
* **Implemente as etapas de treinamento dentro do loop:**
    * Realize estas cinco etapas **em ordem** em cada iteração:
        * **Forward Pass (Passo à Frente)**: Passe seu tensor de `features` para o `model` para obter as previsões.
        * **Calcular a Perda (Loss)**: Use sua `loss_function` para comparar as previsões do modelo com os alvos reais (`targets`).
        * **Zerar Gradientes**: Zere os gradientes no `optimizer` da iteração anterior.
        * **Backward Pass (Passo de Volta)**: Realize a retropropagação na sua `loss` para calcular os novos gradientes.
        * **Atualizar Pesos**: Dê um passo com o `optimizer` para atualizar os parâmetros do modelo.
 
<details>
  <summary><b><font color="green">Dicas Adicionais de Código (Clique para expandir se estiver travado)</font></b></summary>
  
**Para a Inicialização:**
* A função `init_model()` retorna três valores. Você pode desempacotá-los diretamente em suas três variáveis: `model, optimizer, loss_function = init_model()`.

**Para o Forward Pass:**
* Para obter as previsões, você pode chamar seu objeto `model` como uma "função", passando as `features` como o "argumento", `funcao(argumento)`.

**Para Calcular a Perda:**
* A função de perda também funciona como uma função. Ela recebe dois argumentos: suas "previsões" e os "alvos reais".

**Para as Etapas de Gradiente:**
* As três etapas relacionadas ao gradiente (`.zero_grad()`, `.backward()`, e `.step()`) são todos métodos que precisam ser chamados com parênteses, por exemplo, `optimizer.zero_grad()`.

</details>

In [ ]:
# FUNÇÃO AVALIADA: train_model

def train_model(features, targets, epochs, verbose=True):
    """
    Treina o modelo usando os dados fornecidos por um determinado número de épocas.
    
    Argumentos:
        features (torch.Tensor): Os atributos de entrada para o treinamento.
        targets (torch.Tensor): Os valores de alvo para o treinamento.
        epochs (int): O número de épocas de treinamento.
        verbose (bool): Se verdadeiro (True), exibe o progresso do treinamento. Padrão é True.
        
    Retorna:
        model (nn.Sequential): O modelo treinado.
        losses (list): Uma lista de valores de perda (loss) registrados a cada 5000 épocas.
    """
    
    # Inicializa uma lista para armazenar a perda (loss)
    losses = []
    
    ### INICIE O CÓDIGO AQUI ###
    
    # Inicializa o modelo, otimizador e função de perda usando `init_model`
    model, optimizer, loss_function = None

    # Loop através do número especificado de épocas
    for None in range(None)
        
        # Passo para frente (Forward pass): Realiza as previsões
        outputs = None

        # Calcula a perda (loss)
        loss = None

        # Zera os gradientes
        None

        # Passo para trás (Backward pass): Computa os gradientes
        None

        # Atualiza os parâmetros do modelo
        None
    
    ### FIM DO CÓDIGO AQUI ### 

        # A cada 5000 épocas, registra a perda e exibe o progresso
        if (epoch + 1) % 5000 == 0:
            losses.append(loss.item())
            if verbose:
                print(f"Época [{epoch+1}/{epochs}], Perda (Loss): {loss.item():.4f}")
    
    return model, losses

In [ ]:
test_model, loss = train_model(features, targets, 10000)

#### Saída Esperada (approximately):

```
Epoch [5000/10000], Loss: 3.0901
Epoch [10000/10000], Loss: 1.6064
```

In [ ]:
# Teste seu código!
unittests.exercise_4(train_model, features, targets)

<br>

É hora de colocar sua função `train_model` para trabalhar. Execute o treinamento completo nos tensores de `features` e `targets`. Você treinará o modelo por `30.000` épocas (mais do que na execução de teste para garantir a convergência total no conjunto de dados completo), o que dará a ele ampla oportunidade de aprender os padrões nos dados.

In [ ]:
# Loop de treino
model, loss = train_model(features, targets, 30000)

<a name='4'></a>
## 4 - Avaliando o Desempenho do Modelo

Agora que seu modelo está treinado, é hora de avaliar o seu desempenho. Uma maneira simples, porém poderosa, de fazer isso para uma tarefa de regressão é plotar as previsões do modelo em relação aos valores reais (alvos).

* Primeiro, use seu `model` treinado para obter as previsões para todo o conjunto de dados.
* Em seguida, crie um gráfico de dispersão: **Tempos de Entrega Reais (eixo x) vs. Tempos de Entrega Previstos (eixo y)**.
* Se o modelo for preciso, os pontos no gráfico devem formar um agrupamento denso ao longo de uma linha diagonal reta.
    * Quanto mais próximos os pontos estiverem dessa linha, melhores serão as previsões do seu modelo.

Vamos ver quão bem o seu modelo se saiu!

In [ ]:
# Disable gradient calculation for efficient predictions
with torch.no_grad():
    # Perform a forward pass to get model predictions
    predicted_outputs = model(features)

# Plot predictions vs. actual targets to evaluate performance
helper_utils_2.plot_model_predictions(predicted_outputs, targets)

<br>

Como você pode ver no gráfico "Real vs. Previsto", as previsões do modelo (os pontos em cinza claro) formam um agrupamento muito denso que segue a linha de "Previsão Perfeita" quase exatamente. Isso indica que seu modelo aprendeu muito bem os padrões nos dados e está fazendo previsões altamente precisas.

Um resultado como este em um projeto do mundo real seria considerado um grande sucesso. Com o desempenho do seu modelo avaliado, você está pronto para a etapa final: usá-lo para fazer uma previsão em dados novos e inéditos.

<a name='5'></a>
## 5 - Realizando uma Nova Predição

Com um modelo bem treinado e avaliado, você chegou à etapa final e mais prática: a **predição**. É hora de usar seu modelo para fazer uma previsão em dados novos e inéditos.

* Defina um novo cenário de entrega configurando a distância, o horário do dia e se é um final de semana.

**Nota sobre as Regras de Negócio:**
Lembre-se das restrições do serviço de entrega ao definir seus valores:
* **Distância**: Deve ser menor ou igual a `20` milhas.
* **Horário**: Deve estar entre `8.0` (08:00) e `20.0` (20:00).
* **Final de Semana**: Pode ser definido usando `True`/`False` ou `1`/`0`.

In [ ]:
# CÉLULA EDITÁVEL: Defina seus valores abaixo

# Altere os valores abaixo para obter uma estimativa de uma entrega diferente
# Defina a distância para a entrega em milhas
distance_miles = None 
# Defina a hora do dia no formato 24 horas (ex: 9.5 para 9h30)
time_of_day_hours = None
# Use True/False ou 1/0 para indicar se é final de semana
is_weekend = None

# Converte as entradas brutas em um tensor 2D para o modelo
raw_input_tensor = torch.tensor([[distance_miles, time_of_day_hours, is_weekend]], dtype=torch.float32)

Agora, você passará seu `model` treinado, o `data_df` original, seu `raw_input_tensor` e a função `rush_hour_feature` para a função auxiliar. Isso processará suas novas entradas e usará o modelo para gerar o tempo de entrega estimado.

In [ ]:
helper_utils_2.prediction(model, data_df, raw_input_tensor, rush_hour_feature)

## Conclusão

Você passou por todas as etapas fundamentais do pipeline. Você começou com dados brutos de um arquivo `.csv`, realizou a **engenharia de atributos** para adicionar lógica de negócio ao seu conjunto de dados e construiu um **pipeline de preparação de dados** completo para automatizar o processo.

A partir daí, você projetou e **treinou** uma rede neural de múltiplas camadas, indo além dos modelos simples dos laboratórios anteriores. Em seguida, **avaliou** o desempenho do modelo visualizando suas previsões e, por fim, utilizou seu modelo treinado para fazer uma **predição** em dados novos e inéditos.

As habilidades que você praticou aqui — manipular tensores, projetar atributos e construir pipelines de treinamento de ponta a ponta — são os blocos fundamentais para enfrentar desafios ainda mais complexos em deep learning. Você agora tem uma base sólida para continuar progredindo. Bom trabalho!